# Huffman paso a paso

Este cuaderno construye un código de Huffman para una fuente discreta pequeña.

Huffman produce un código prefijo óptimo cuando codificamos símbolo a símbolo con probabilidades conocidas.

In [ ]:
import heapq
import itertools
import math


def entropia(distribucion):
    return -sum(p * math.log2(p) for p in distribucion.values() if p > 0)


def construir_huffman(distribucion):
    contador = itertools.count()
    heap = [(p, next(contador), simbolo) for simbolo, p in distribucion.items()]
    heapq.heapify(heap)
    pasos = []
    while len(heap) > 1:
        p1, _, a = heapq.heappop(heap)
        p2, _, b = heapq.heappop(heap)
        nodo = (a, b)
        pasos.append((p1, a, p2, b, p1 + p2))
        heapq.heappush(heap, (p1 + p2, next(contador), nodo))
    return heap[0][2], pasos


def extraer_codigos(arbol, prefijo=""):
    if isinstance(arbol, str):
        return {arbol: prefijo or "0"}
    izquierda, derecha = arbol
    codigos = {}
    codigos.update(extraer_codigos(izquierda, prefijo + "0"))
    codigos.update(extraer_codigos(derecha, prefijo + "1"))
    return codigos


def longitud_media(distribucion, codigo):
    return sum(distribucion[s] * len(codigo[s]) for s in distribucion)

## Construcción del árbol

En cada paso se combinan los dos símbolos o subárboles menos probables.

In [ ]:
fuente = {"A": 0.4, "B": 0.25, "C": 0.2, "D": 0.1, "E": 0.05}
arbol, pasos = construir_huffman(fuente)
codigo = extraer_codigos(arbol)

for i, paso in enumerate(pasos, 1):
    p1, a, p2, b, total = paso
    print(f"paso {i}: combinar {a} ({p1}) + {b} ({p2}) -> {total}")

print("\nCódigo resultante:")
for simbolo, palabra in sorted(codigo.items()):
    print(f"{simbolo}: {palabra}")

## Comparación con la entropía

La longitud media de Huffman debe quedar por encima de la entropía, pero cerca de ella.

In [ ]:
h = entropia(fuente)
l = longitud_media(fuente, codigo)

print(f"H(X) = {h:.4f} bits/símbolo")
print(f"L    = {l:.4f} bits/símbolo")
print(f"redundancia L-H = {l - h:.4f} bits/símbolo")

## Codificar y decodificar

Como el código es prefijo, la decodificación puede hacerse de izquierda a derecha.

In [ ]:
def codificar(mensaje, codigo):
    return "".join(codigo[s] for s in mensaje)


def decodificar(bits, codigo):
    inverso = {palabra: simbolo for simbolo, palabra in codigo.items()}
    salida = []
    buffer = ""
    for bit in bits:
        buffer += bit
        if buffer in inverso:
            salida.append(inverso[buffer])
            buffer = ""
    if buffer:
        raise ValueError("Quedaron bits sin decodificar.")
    return salida


mensaje = list("ABACABADAE")
bits = codificar(mensaje, codigo)
print("mensaje:", mensaje)
print("bits:", bits)
print("decodificado:", decodificar(bits, codigo))

## Para experimentar

1. Cambia las probabilidades de la fuente.
2. Añade un símbolo nuevo poco probable.
3. Compara la longitud media de Huffman con un código fijo.